In [3]:
"""
PREPROCESSING DATA PM2.5 - TAHAP 1 BAGIAN 1-4
============================================================

Mencakup:
1. Membaca data CSV/Excel
2. Mengisi missing timestamp dengan interval 1 menit
3. Menangani missing value
4. Quality control berbasis batas fisik

Kolom yang diharapkan:
- waktu
- PM10
- PM2_5
- humidity
- pressure
- solar_radiation
- temperature
- total_ch
- wind_direction
- wind_speed

Cara menjalankan:
    python preprocessing_part_1_4.py

Output:
- data/preprocessed/data_preprocessed_part_1_4.csv
- data/preprocessed/missing_timestamp_report.csv
- data/preprocessed/missing_value_report.csv
- data/preprocessed/quality_control_report.csv
- data/preprocessed/preprocessing_summary.txt
"""

from __future__ import annotations

import logging
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


# ============================================================
# 1. KONFIGURASI
# ============================================================

@dataclass
class PreprocessingConfig:
    """Konfigurasi utama preprocessing."""

    # Lokasi file masukan.
    input_file: str = "datady.csv"

    # Folder keluaran.
    output_directory: str = "."

    # Nama kolom waktu.
    timestamp_column: str = "waktu"

    # Resolusi data.
    frequency: str = "1min"

    # Gap maksimum yang boleh diinterpolasi.
    max_interpolation_gap_minutes: int = 30

    # Kolom kontinu yang boleh diinterpolasi.
    continuous_columns: List[str] = field(
        default_factory=lambda: [
            "PM10",
            "PM2_5",
            "humidity",
            "pressure",
            "solar_radiation",
            "temperature",
            "wind_direction",
            "wind_speed",
        ]
    )

    # Kolom yang tidak diinterpolasi secara otomatis.
    # total_ch diperlakukan konservatif karena belum diketahui
    # apakah merupakan curah hujan per interval atau akumulasi.
    non_interpolated_columns: List[str] = field(
        default_factory=lambda: [
            "total_ch",
        ]
    )

    # Batas quality control.
    # Nilai di luar rentang ini akan diubah menjadi NaN.
    qc_limits: Dict[str, Tuple[float, float]] = field(
        default_factory=lambda: {
            "PM10": (0.0, 1000.0),
            "PM2_5": (0.0, 1000.0),
            "humidity": (0.0, 100.0),
            "pressure": (850.0, 1100.0),
            "solar_radiation": (0.0, 1600.0),
            "temperature": (-20.0, 60.0),
            "total_ch": (0.0, 500.0),
            "wind_direction": (0.0, 360.0),
            "wind_speed": (0.0, 75.0),
        }
    )

    # Jika ada timestamp duplikat, data numerik akan dirata-ratakan.
    duplicate_aggregation: str = "mean"


CONFIG = PreprocessingConfig()


# ============================================================
# 2. LOGGING
# ============================================================

def setup_logger(output_directory: Path) -> logging.Logger:
    """Membuat logger untuk terminal dan file."""

    output_directory.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger("pm25_preprocessing")
    logger.setLevel(logging.INFO)

    # Menghindari duplikasi handler saat script dijalankan ulang.
    if logger.handlers:
        return logger

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(
        output_directory / "preprocessing.log",
        mode="w",
        encoding="utf-8",
    )
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

    return logger


# ============================================================
# 3. UTILITAS NAMA KOLOM
# ============================================================

def normalize_column_name(column_name: str) -> str:
    """
    Menormalkan variasi nama kolom tanpa mengubah nama standar penelitian.

    Contoh:
    PM2.5 -> PM2_5
    pm2 5 -> PM2_5
    Humidity -> humidity
    """

    normalized = str(column_name).strip()
    normalized = normalized.replace(".", "_")
    normalized = normalized.replace("-", "_")
    normalized = normalized.replace(" ", "_")

    while "__" in normalized:
        normalized = normalized.replace("__", "_")

    key = normalized.lower()

    aliases = {
        "waktu": "waktu",
        "time": "waktu",
        "timestamp": "waktu",
        "datetime": "waktu",
        "date_time": "waktu",

        "pm10": "PM10",
        "pm_10": "PM10",

        "pm2_5": "PM2_5",
        "pm25": "PM2_5",
        "pm_2_5": "PM2_5",

        "humidity": "humidity",
        "rh": "humidity",
        "relative_humidity": "humidity",

        "pressure": "pressure",
        "air_pressure": "pressure",
        "barometric_pressure": "pressure",

        "solar_radiation": "solar_radiation",
        "solar": "solar_radiation",
        "radiation": "solar_radiation",

        "temperature": "temperature",
        "temp": "temperature",
        "air_temperature": "temperature",

        "total_ch": "total_ch",
        "rainfall": "total_ch",
        "precipitation": "total_ch",
        "rain": "total_ch",

        "wind_direction": "wind_direction",
        "wd": "wind_direction",
        "wind_dir": "wind_direction",

        "wind_speed": "wind_speed",
        "ws": "wind_speed",
    }

    return aliases.get(key, normalized)


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Menormalkan seluruh nama kolom."""

    renamed_columns = {
        column: normalize_column_name(column)
        for column in df.columns
    }

    return df.rename(columns=renamed_columns)


# ============================================================
# 4. MEMBACA DATA
# ============================================================

def read_input_data(
    file_path: Path,
    timestamp_column: str,
    logger: logging.Logger,
) -> pd.DataFrame:
    """
    Membaca file CSV atau Excel dan melakukan validasi awal.
    """

    if not file_path.exists():
        raise FileNotFoundError(
            f"File input tidak ditemukan: {file_path.resolve()}"
        )

    logger.info("Membaca file: %s", file_path)

    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        # sep=None memungkinkan pandas mendeteksi koma, titik koma,
        # atau delimiter lain secara otomatis.
        df = pd.read_csv(
            file_path,
            sep=None,
            engine="python",
            encoding="utf-8-sig",
        )

    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(file_path)

    else:
        raise ValueError(
            "Format file tidak didukung. Gunakan CSV, XLSX, atau XLS."
        )

    if df.empty:
        raise ValueError("Dataset kosong.")

    df = normalize_columns(df)

    if timestamp_column not in df.columns:
        raise KeyError(
            f"Kolom waktu '{timestamp_column}' tidak ditemukan.\n"
            f"Kolom yang tersedia: {list(df.columns)}"
        )

    logger.info("Jumlah baris awal: %s", f"{len(df):,}")
    logger.info("Kolom terdeteksi: %s", list(df.columns))

    return df


def convert_data_types(
    df: pd.DataFrame,
    timestamp_column: str,
    logger: logging.Logger,
) -> pd.DataFrame:
    """
    Mengubah waktu menjadi datetime dan parameter sensor menjadi numerik.
    """

    result = df.copy()

    # dayfirst=False sesuai format YYYY-MM-DD.
    result[timestamp_column] = pd.to_datetime(
        result[timestamp_column],
        errors="coerce",
        dayfirst=False,
    )

    invalid_timestamp_count = result[timestamp_column].isna().sum()

    if invalid_timestamp_count > 0:
        logger.warning(
            "%s baris memiliki timestamp tidak valid dan akan dihapus.",
            invalid_timestamp_count,
        )

        invalid_timestamp_rows = result[
            result[timestamp_column].isna()
        ].copy()

        invalid_timestamp_rows.to_csv(
            Path(CONFIG.output_directory) / "invalid_timestamp_rows.csv",
            index=False,
        )

        result = result.dropna(subset=[timestamp_column])

    sensor_columns = [
        column
        for column in result.columns
        if column != timestamp_column
    ]

    for column in sensor_columns:
        # Mengatasi angka desimal yang mungkin memakai koma.
        if result[column].dtype == "object":
            result[column] = (
                result[column]
                .astype(str)
                .str.strip()
                .str.replace(",", ".", regex=False)
            )

        result[column] = pd.to_numeric(
            result[column],
            errors="coerce",
        )

    return result


def handle_duplicate_timestamps(
    df: pd.DataFrame,
    timestamp_column: str,
    aggregation: str,
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, int]:
    """
    Menangani timestamp duplikat.

    Default:
    Semua nilai numerik pada timestamp yang sama dirata-ratakan.
    """

    duplicate_mask = df.duplicated(
        subset=[timestamp_column],
        keep=False,
    )

    duplicate_count = int(duplicate_mask.sum())

    if duplicate_count == 0:
        logger.info("Tidak ditemukan timestamp duplikat.")
        return df, 0

    logger.warning(
        "Ditemukan %s baris yang memiliki timestamp duplikat.",
        duplicate_count,
    )

    duplicate_rows = df.loc[duplicate_mask].sort_values(
        timestamp_column
    )

    duplicate_rows.to_csv(
        Path(CONFIG.output_directory) / "duplicate_timestamp_rows.csv",
        index=False,
    )

    if aggregation == "mean":
        result = (
            df.groupby(timestamp_column, as_index=False)
            .mean(numeric_only=True)
        )

    elif aggregation == "first":
        result = df.drop_duplicates(
            subset=[timestamp_column],
            keep="first",
        )

    elif aggregation == "last":
        result = df.drop_duplicates(
            subset=[timestamp_column],
            keep="last",
        )

    else:
        raise ValueError(
            f"Metode agregasi duplikat tidak dikenali: {aggregation}"
        )

    return result, duplicate_count


# ============================================================
# 5. MELENGKAPI MISSING TIMESTAMP
# ============================================================

def complete_timestamp_index(
    df: pd.DataFrame,
    timestamp_column: str,
    frequency: str,
    output_directory: Path,
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Membuat indeks waktu lengkap pada resolusi 1 menit.

    Timestamp yang sebelumnya tidak tersedia akan ditambahkan
    sebagai baris baru dengan nilai parameter NaN.
    """

    result = df.copy()

    result = result.sort_values(timestamp_column)
    result = result.set_index(timestamp_column)

    original_index = result.index.copy()

    full_index = pd.date_range(
        start=result.index.min(),
        end=result.index.max(),
        freq=frequency,
        name=timestamp_column,
    )

    missing_timestamps = full_index.difference(original_index)

    missing_timestamp_report = pd.DataFrame(
        {timestamp_column: missing_timestamps}
    )

    missing_timestamp_report.to_csv(
        output_directory / "missing_timestamp_report.csv",
        index=False,
    )

    result = result.reindex(full_index)

    # Penanda apakah baris berasal dari timestamp asli atau ditambahkan.
    result["timestamp_was_missing"] = (
        ~result.index.isin(original_index)
    ).astype("int8")

    logger.info(
        "Rentang data: %s hingga %s",
        result.index.min(),
        result.index.max(),
    )

    logger.info(
        "Jumlah timestamp yang ditambahkan: %s",
        f"{len(missing_timestamps):,}",
    )

    logger.info(
        "Jumlah baris setelah reindex: %s",
        f"{len(result):,}",
    )

    return result, missing_timestamp_report


# ============================================================
# 6. QUALITY CONTROL
# ============================================================

def apply_range_quality_control(
    df: pd.DataFrame,
    qc_limits: Dict[str, Tuple[float, float]],
    output_directory: Path,
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Mengubah nilai di luar rentang QC menjadi NaN.

    QC dilakukan sebelum imputasi agar nilai tidak valid
    tidak dipakai sebagai titik acuan interpolasi.
    """

    result = df.copy()
    qc_records = []

    for column, limits in qc_limits.items():
        minimum, maximum = limits

        if column not in result.columns:
            logger.warning(
                "Kolom QC '%s' tidak tersedia dan dilewati.",
                column,
            )
            continue

        invalid_mask = (
            result[column].notna()
            & (
                (result[column] < minimum)
                | (result[column] > maximum)
            )
        )

        invalid_count = int(invalid_mask.sum())

        # Flag QC per variabel.
        flag_column = f"{column}_qc_flag"
        result[flag_column] = invalid_mask.astype("int8")

        qc_records.append(
            {
                "parameter": column,
                "minimum_allowed": minimum,
                "maximum_allowed": maximum,
                "invalid_count": invalid_count,
                "invalid_percentage": (
                    invalid_count / len(result) * 100
                    if len(result) > 0
                    else 0
                ),
            }
        )

        if invalid_count > 0:
            logger.warning(
                "%s: %s nilai di luar rentang [%s, %s] "
                "diubah menjadi NaN.",
                column,
                f"{invalid_count:,}",
                minimum,
                maximum,
            )

            result.loc[invalid_mask, column] = np.nan
        else:
            logger.info(
                "%s: seluruh nilai berada dalam rentang QC.",
                column,
            )

    qc_report = pd.DataFrame(qc_records)

    qc_report.to_csv(
        output_directory / "quality_control_report.csv",
        index=False,
    )

    return result, qc_report


# ============================================================
# 7. IDENTIFIKASI GAP MISSING VALUE
# ============================================================

def calculate_missing_gap_length(series: pd.Series) -> pd.Series:
    """
    Menghitung panjang setiap kelompok missing value.

    Contoh:
        nilai : 10, NaN, NaN, 14
        gap   :  0,   2,   2,  0
    """

    missing_mask = series.isna()

    group_id = (
        missing_mask.ne(missing_mask.shift())
        .cumsum()
    )

    gap_length = (
        missing_mask
        .groupby(group_id)
        .transform("sum")
        .where(missing_mask, 0)
    )

    return gap_length.astype("int32")


# ============================================================
# 8. MENANGANI MISSING VALUE
# ============================================================

def interpolate_circular_wind_direction(
    series: pd.Series,
    max_gap: int,
) -> pd.Series:
    """
    Interpolasi khusus arah angin.

    Arah angin merupakan data siklik:
    359 derajat dekat dengan 1 derajat.

    Interpolasi langsung dapat menghasilkan nilai salah,
    sehingga arah diubah ke komponen sinus dan kosinus terlebih dahulu.
    """

    radians = np.deg2rad(series)

    sin_component = pd.Series(
        np.sin(radians),
        index=series.index,
    )

    cos_component = pd.Series(
        np.cos(radians),
        index=series.index,
    )

    interpolated_sin = sin_component.interpolate(
        method="time",
        limit=max_gap,
        limit_area="inside",
    )

    interpolated_cos = cos_component.interpolate(
        method="time",
        limit=max_gap,
        limit_area="inside",
    )

    interpolated_direction = (
        np.rad2deg(
            np.arctan2(
                interpolated_sin,
                interpolated_cos,
            )
        )
        + 360
    ) % 360

    # Nilai asli tetap dipertahankan.
    result = series.copy()
    result.loc[series.isna()] = interpolated_direction.loc[
        series.isna()
    ]

    return result


def handle_missing_values(
    df: pd.DataFrame,
    continuous_columns: List[str],
    non_interpolated_columns: List[str],
    max_gap_minutes: int,
    output_directory: Path,
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Menangani missing value dengan aturan:

    - Gap 1 sampai max_gap_minutes:
      diinterpolasi menggunakan metode waktu.
    - Gap lebih panjang:
      dibiarkan NaN.
    - total_ch:
      tidak diinterpolasi secara otomatis.
    """

    result = df.copy()
    missing_records = []

    for column in continuous_columns:
        if column not in result.columns:
            logger.warning(
                "Kolom '%s' tidak ditemukan dan tidak diimputasi.",
                column,
            )
            continue

        missing_before = int(result[column].isna().sum())

        # Menghitung panjang gap sebelum interpolasi.
        gap_length = calculate_missing_gap_length(
            result[column]
        )

        result[f"{column}_missing_flag"] = (
            result[column].isna()
        ).astype("int8")

        result[f"{column}_gap_length"] = gap_length

        original_series = result[column].copy()

        if column == "wind_direction":
            interpolated = interpolate_circular_wind_direction(
                result[column],
                max_gap=max_gap_minutes,
            )
        else:
            interpolated = result[column].interpolate(
                method="time",
                limit=max_gap_minutes,
                limit_area="inside",
            )

        # Hanya gap dengan panjang total <= batas yang boleh diisi.
        allowed_gap_mask = (
            original_series.isna()
            & (gap_length <= max_gap_minutes)
        )

        result.loc[allowed_gap_mask, column] = interpolated.loc[
            allowed_gap_mask
        ]

        imputed_mask = (
            original_series.isna()
            & result[column].notna()
        )

        result[f"{column}_imputed_flag"] = (
            imputed_mask.astype("int8")
        )

        missing_after = int(result[column].isna().sum())
        imputed_count = int(imputed_mask.sum())

        long_gap_count = int(
            (
                original_series.isna()
                & (gap_length > max_gap_minutes)
            ).sum()
        )

        missing_records.append(
            {
                "parameter": column,
                "missing_before": missing_before,
                "imputed_count": imputed_count,
                "missing_after": missing_after,
                "long_gap_points_not_imputed": long_gap_count,
                "max_interpolation_gap_minutes": (
                    max_gap_minutes
                ),
            }
        )

        logger.info(
            "%s | missing awal: %s | diisi: %s | "
            "tersisa: %s",
            column,
            f"{missing_before:,}",
            f"{imputed_count:,}",
            f"{missing_after:,}",
        )

    # Variabel non-interpolasi tetap diberi flag.
    for column in non_interpolated_columns:
        if column not in result.columns:
            continue

        missing_before = int(result[column].isna().sum())
        gap_length = calculate_missing_gap_length(
            result[column]
        )

        result[f"{column}_missing_flag"] = (
            result[column].isna()
        ).astype("int8")

        result[f"{column}_gap_length"] = gap_length
        result[f"{column}_imputed_flag"] = 0

        missing_records.append(
            {
                "parameter": column,
                "missing_before": missing_before,
                "imputed_count": 0,
                "missing_after": missing_before,
                "long_gap_points_not_imputed": int(
                    result[column].isna().sum()
                ),
                "max_interpolation_gap_minutes": 0,
            }
        )

        logger.info(
            "%s tidak diinterpolasi secara otomatis. "
            "Missing tersisa: %s",
            column,
            f"{missing_before:,}",
        )

    missing_report = pd.DataFrame(missing_records)

    missing_report.to_csv(
        output_directory / "missing_value_report.csv",
        index=False,
    )

    return result, missing_report


# ============================================================
# 9. PEMERIKSAAN KONSISTENSI ANTARVARIABEL
# ============================================================

def apply_cross_variable_quality_control(
    df: pd.DataFrame,
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Pemeriksaan konsistensi sederhana antarparameter.

    Aturan:
    PM2.5 seharusnya tidak jauh lebih besar daripada PM10.

    Karena sensor low-cost dapat memiliki noise,
    digunakan toleransi 20 persen atau minimum 5 µg/m³.
    """

    result = df.copy()
    records = []

    if {"PM2_5", "PM10"}.issubset(result.columns):
        tolerance = np.maximum(
            0.20 * result["PM10"],
            5.0,
        )

        inconsistent_mask = (
            result["PM2_5"].notna()
            & result["PM10"].notna()
            & (
                result["PM2_5"]
                > result["PM10"] + tolerance
            )
        )

        inconsistent_count = int(
            inconsistent_mask.sum()
        )

        result["PM_consistency_qc_flag"] = (
            inconsistent_mask.astype("int8")
        )

        records.append(
            {
                "quality_control_rule": (
                    "PM2_5 <= PM10 + tolerance"
                ),
                "flagged_count": inconsistent_count,
                "action": "flag only",
            }
        )

        logger.info(
            "Konsistensi PM2.5-PM10: %s titik ditandai.",
            f"{inconsistent_count:,}",
        )

        # Nilai tidak langsung dihapus.
        # Flag akan dianalisis lebih lanjut pada tahap outlier.
    else:
        logger.warning(
            "Pemeriksaan PM2.5-PM10 dilewati karena kolom "
            "tidak lengkap."
        )

    consistency_report = pd.DataFrame(records)

    return result, consistency_report


# ============================================================
# 10. RINGKASAN DATA
# ============================================================

def create_dataset_summary(
    original_df: pd.DataFrame,
    processed_df: pd.DataFrame,
    duplicate_count: int,
    missing_timestamp_count: int,
    output_directory: Path,
) -> None:
    """Menyimpan ringkasan proses dalam file TXT."""

    summary_lines = [
        "RINGKASAN PREPROCESSING PM2.5",
        "=" * 60,
        "",
        f"Jumlah baris data awal       : {len(original_df):,}",
        f"Jumlah baris data akhir      : {len(processed_df):,}",
        f"Timestamp duplikat ditemukan : {duplicate_count:,}",
        (
            "Timestamp hilang ditambahkan: "
            f"{missing_timestamp_count:,}"
        ),
        "",
        "Rentang waktu:",
        f"Awal  : {processed_df.index.min()}",
        f"Akhir : {processed_df.index.max()}",
        "",
        "Missing value akhir:",
    ]

    sensor_columns = [
        column
        for column in CONFIG.qc_limits
        if column in processed_df.columns
    ]

    for column in sensor_columns:
        missing_count = int(
            processed_df[column].isna().sum()
        )

        missing_percentage = (
            missing_count / len(processed_df) * 100
            if len(processed_df) > 0
            else 0
        )

        summary_lines.append(
            f"- {column:18s}: "
            f"{missing_count:8,d} "
            f"({missing_percentage:6.2f}%)"
        )

    summary_file = (
        output_directory / "preprocessing_summary.txt"
    )

    summary_file.write_text(
        "\n".join(summary_lines),
        encoding="utf-8",
    )


# ============================================================
# 11. PIPELINE UTAMA
# ============================================================

def run_preprocessing(
    config: PreprocessingConfig,
) -> pd.DataFrame:
    """
    Menjalankan seluruh preprocessing bagian 1-4.
    """

    input_path = Path(config.input_file)
    output_directory = Path(config.output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)

    logger = setup_logger(output_directory)

    logger.info("=" * 70)
    logger.info("MEMULAI PREPROCESSING PM2.5 BAGIAN 1-4")
    logger.info("=" * 70)

    # --------------------------------------------------------
    # Tahap 1: Membaca data
    # --------------------------------------------------------
    raw_df = read_input_data(
        file_path=input_path,
        timestamp_column=config.timestamp_column,
        logger=logger,
    )

    typed_df = convert_data_types(
        df=raw_df,
        timestamp_column=config.timestamp_column,
        logger=logger,
    )

    deduplicated_df, duplicate_count = (
        handle_duplicate_timestamps(
            df=typed_df,
            timestamp_column=config.timestamp_column,
            aggregation=config.duplicate_aggregation,
            logger=logger,
        )
    )

    # --------------------------------------------------------
    # Tahap 2: Mengisi missing timestamp
    # --------------------------------------------------------
    complete_df, missing_timestamp_report = (
        complete_timestamp_index(
            df=deduplicated_df,
            timestamp_column=config.timestamp_column,
            frequency=config.frequency,
            output_directory=output_directory,
            logger=logger,
        )
    )

    # --------------------------------------------------------
    # Tahap 4 dijalankan sebelum imputasi.
    # Nilai tidak valid tidak boleh menjadi acuan interpolasi.
    # --------------------------------------------------------
    qc_df, qc_report = apply_range_quality_control(
        df=complete_df,
        qc_limits=config.qc_limits,
        output_directory=output_directory,
        logger=logger,
    )

    qc_df, consistency_report = (
        apply_cross_variable_quality_control(
            df=qc_df,
            logger=logger,
        )
    )

    if not consistency_report.empty:
        consistency_report.to_csv(
            output_directory
            / "cross_variable_quality_control_report.csv",
            index=False,
        )

    # --------------------------------------------------------
    # Tahap 3: Menangani missing value
    # --------------------------------------------------------
    processed_df, missing_value_report = (
        handle_missing_values(
            df=qc_df,
            continuous_columns=config.continuous_columns,
            non_interpolated_columns=(
                config.non_interpolated_columns
            ),
            max_gap_minutes=(
                config.max_interpolation_gap_minutes
            ),
            output_directory=output_directory,
            logger=logger,
        )
    )

    # Membuat penanda apakah suatu baris masih memiliki
    # minimal satu missing value pada parameter utama.
    main_columns = [
        column
        for column in config.qc_limits
        if column in processed_df.columns
    ]

    processed_df["has_remaining_missing_value"] = (
        processed_df[main_columns]
        .isna()
        .any(axis=1)
        .astype("int8")
    )

    # Menyimpan data.
    output_csv = (
        output_directory
        / "data_preprocessed_part_1_4.csv"
    )

    processed_df.to_csv(
        output_csv,
        index=True,
        index_label=config.timestamp_column,
    )

    # Parquet lebih cepat untuk tahap deep learning.
    output_parquet = (
        output_directory
        / "data_preprocessed_part_1_4.parquet"
    )

    try:
        processed_df.to_parquet(output_parquet)
        logger.info(
            "File Parquet disimpan: %s",
            output_parquet,
        )
    except ImportError:
        logger.warning(
            "File Parquet tidak dibuat karena pyarrow belum "
            "terinstal. CSV tetap berhasil dibuat."
        )

    create_dataset_summary(
        original_df=raw_df,
        processed_df=processed_df,
        duplicate_count=duplicate_count,
        missing_timestamp_count=len(
            missing_timestamp_report
        ),
        output_directory=output_directory,
    )

    logger.info("=" * 70)
    logger.info("PREPROCESSING BAGIAN 1-4 SELESAI")
    logger.info("Output utama: %s", output_csv)
    logger.info("=" * 70)

    return processed_df


# ============================================================
# 12. EKSEKUSI
# ============================================================

if __name__ == "__main__":
    try:
        final_dataframe = run_preprocessing(CONFIG)

        print("\nLima baris pertama hasil preprocessing:")
        print(final_dataframe.head())

        print("\nLima baris terakhir hasil preprocessing:")
        print(final_dataframe.tail())

        print("\nUkuran data akhir:")
        print(final_dataframe.shape)

    except Exception as error:
        logging.exception(
            "Preprocessing gagal: %s",
            error,
        )
        raise

ERROR:root:Preprocessing gagal: [Errno 30] Read-only file system: '/preprocessing.log'
Traceback (most recent call last):
  File "/var/folders/n9/vvxfv1ts0bsdgxy7kmr426m80000gn/T/ipykernel_64255/2602835980.py", line 1122, in <module>
    final_dataframe = run_preprocessing(CONFIG)
  File "/var/folders/n9/vvxfv1ts0bsdgxy7kmr426m80000gn/T/ipykernel_64255/2602835980.py", line 967, in run_preprocessing
    logger = setup_logger(output_directory)
  File "/var/folders/n9/vvxfv1ts0bsdgxy7kmr426m80000gn/T/ipykernel_64255/2602835980.py", line 139, in setup_logger
    file_handler = logging.FileHandler(
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/logging/__init__.py", line 1146, in __init__
    StreamHandler.__init__(self, self._open())
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/logging/__init__.py", line 1175, in _open
    return open(self.baseFilename, self.mode, enc

OSError: [Errno 30] Read-only file system: '/preprocessing.log'